# 02 — TFNE forward-field tutorial

TFNE is used here as a forward model for current density, CSD, and LFP-like readouts.  This notebook starts with a plain medium and source conservation before any cortical-column claim.


In [1]:
import os, sys, math, json
from pathlib import Path
import numpy as np

SMOKE_MODE = True
SEED = 7
np.random.seed(SEED)

# Allow running from repo root or from the tutorials directory.
ROOT = Path.cwd()
if not (ROOT / "src" / "jbiophysic").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("ROOT", ROOT)
print("SMOKE_MODE", SMOKE_MODE, "SEED", SEED)


ROOT /mnt/data/work_merge/jbiophysic-main
SMOKE_MODE True SEED 7


In [2]:
import jax.numpy as jnp
from jbiophysic.tfne.fields import make_regular_grid, initialize_potentials, mean_zero_gauge
from jbiophysic.tfne.sources import project_sparse_currents, integrate_source, conservation_error
from jbiophysic.tfne.tensors import isotropic_gamma, tensor_eigenvalue_diagnostics
from jbiophysic.tfne.csd import current_density, divergence_neumann_zero


## Grid, gauge, and passive tensor


In [3]:
grid = make_regular_grid((12, 12, 12), (100e-6, 100e-6, 100e-6))
Gamma = isotropic_gamma(0.3, grid.shape)
min_eig, max_eig, cond = tensor_eigenvalue_diagnostics(Gamma)
phi_e = mean_zero_gauge(jnp.zeros(grid.shape), grid.active_mask)
V_m = jnp.ones(grid.shape) * -70e-3
phi_i, phi_e, V_m = initialize_potentials(phi_e, V_m, active_mask=grid.active_mask)
print({"grid": grid.shape, "voxel_volume": grid.voxel_volume, "gamma_min": float(min_eig), "gamma_cond": float(cond), "phi_e_mean": float(jnp.mean(phi_e))})


{'grid': (12, 12, 12), 'voxel_volume': 1e-12, 'gamma_min': 0.30000001192092896, 'gamma_cond': 1.0, 'phi_e_mean': 0.0}


## Mollified sparse source conservation


In [4]:
positions = jnp.asarray([[300e-6, 300e-6, 300e-6], [700e-6, 500e-6, 800e-6]])
currents = jnp.asarray([1e-12, -0.5e-12])
radii = jnp.asarray([150e-6, 150e-6])
q = project_sparse_currents(grid, currents, positions, radii)
target = jnp.sum(currents)
err = conservation_error(grid, q, target)
print({"q_shape": q.shape, "target_A": float(target), "integrated_A": float(integrate_source(grid, q)), "err_A": float(err), "finite": bool(jnp.all(jnp.isfinite(q)))})


{'q_shape': (12, 12, 12), 'target_A': 4.999999980020986e-13, 'integrated_A': 5.000028169277471e-13, 'err_A': 2.8189256484623115e-18, 'finite': True}


## A simple current-density diagnostic


In [5]:
J = current_density(phi_e, Gamma, grid)
divJ = divergence_neumann_zero(J, grid)
print({"J_shape": J.shape, "div_shape": divJ.shape, "J_finite": bool(jnp.all(jnp.isfinite(J))), "div_l2": float(jnp.sqrt(jnp.mean(divJ**2)))})


{'J_shape': (3, 12, 12, 12), 'div_shape': (12, 12, 12), 'J_finite': True, 'div_l2': 0.0}
